# Setup

In [1]:
from pathlib import Path
import sqlite3
import pickle

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
DB_PATH = PROJECT_ROOT / "db" / "app.db"

TOPIC_MODEL_NAME = "kmeans_all-MiniLM-L6-v2_k15"
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print("DB_PATH exists:", DB_PATH.exists(), DB_PATH)
print("TOPIC_MODEL_NAME:", TOPIC_MODEL_NAME)
print("EMBEDDING_MODEL_NAME:", EMBEDDING_MODEL_NAME)

DB_PATH exists: True /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/db/app.db
TOPIC_MODEL_NAME: kmeans_all-MiniLM-L6-v2_k15
EMBEDDING_MODEL_NAME: all-MiniLM-L6-v2


In [2]:
# load topic assignments with metadata
conn = sqlite3.connect(DB_PATH)

topic_docs_df = pd.read_sql_query("""
    SELECT
        d.id AS document_id,
        d.publication_year,
        d.title,
        dt.topic_id,
        t.topic_label
    FROM document_topics dt
    JOIN documents d
        ON dt.document_id = d.id
    JOIN topics t
        ON dt.topic_id = t.topic_id
       AND dt.model_name = t.model_name
    WHERE dt.model_name = ?
    ORDER BY d.publication_year, d.id
""", conn, params=[TOPIC_MODEL_NAME])

conn.close()

print("Loaded topic-assigned documents:", len(topic_docs_df))
display(topic_docs_df.head())

Loaded topic-assigned documents: 738


,document_id,publication_year,title,topic_id,topic_label
0,1,2020,Study protocol: Comparison of different risk p...,6,Epidemiological Modeling & Case Data
1,2,2020,"Mental health, sleep quality and quality of li...",5,Mental Health & Psychological Effects
2,3,2020,Parent perspectives on food allergy management...,9,Food Systems & Hospitality Impact
3,4,2020,COVID-19 critical illness in Sweden: character...,14,Hospitalization & Patient Outcomes
4,5,2020,Geolocated Twitter-based population mobility i...,13,Social Media & Misinformation


# Topics Over Time

In [3]:
# Topic by Year Counts
topic_year_counts = (
    topic_docs_df
    .groupby(["publication_year", "topic_id"])
    .size()
    .unstack(fill_value=0)
    .sort_index()
)

print("Topic-by-year count matrix shape:", topic_year_counts.shape)
display(topic_year_counts)

Topic-by-year count matrix shape: (3, 15)


topic_id,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
publication_year,,,,,,,,,,,,,,,
2020,55,13,22,10,5,38,56,32,36,10,17,46,53,28,29
2021,13,16,23,14,27,15,14,14,30,12,13,15,30,4,21
2022,2,2,2,1,1,3,0,0,2,1,1,2,4,0,6


In [4]:
# Topic Prevalence by Year
topic_year_prevalence = topic_year_counts.div(
    topic_year_counts.sum(axis=1),
    axis=0
)

print("Row sums (should be 1):")
print(topic_year_prevalence.sum(axis=1))

display(topic_year_prevalence)

Row sums (should be 1):
publication_year
2020    1.0
2021    1.0
2022    1.0
dtype: float64


topic_id,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
publication_year,,,,,,,,,,,,,,,
2020,0.122222,0.028889,0.048889,0.022222,0.011111,0.084444,0.124444,0.071111,0.080000,0.022222,0.037778,0.102222,0.117778,0.062222,0.064444
2021,0.049808,0.061303,0.088123,0.053640,0.103448,0.057471,0.053640,0.053640,0.114943,0.045977,0.049808,0.057471,0.114943,0.015326,0.080460
2022,0.074074,0.074074,0.074074,0.037037,0.037037,0.111111,0.000000,0.000000,0.074074,0.037037,0.037037,0.074074,0.148148,0.000000,0.222222


In [5]:
# Mapping topics to labels
topic_id_to_label = (
    topic_docs_df[["topic_id", "topic_label"]]
    .drop_duplicates()
    .set_index("topic_id")["topic_label"]
    .to_dict()
)

topic_year_counts_labeled = topic_year_counts.rename(columns=topic_id_to_label)
topic_year_prevalence_labeled = topic_year_prevalence.rename(columns=topic_id_to_label)

display(topic_year_counts_labeled)
display(topic_year_prevalence_labeled)

topic_id,Public Health & Workforce Impact,Online Education & Training,Clinical Virology & Infection,Medical Imaging & Deep Learning,Financial Markets & Economic Impact,Mental Health & Psychological Effects,Epidemiological Modeling & Case Data,"Digital Health, Telemedicine & Blockchain",Severe Disease & Immune Response,Food Systems & Hospitality Impact,"Prevention, Masks & Environmental Effects","Policy, Governance & Public Response",Clinical Care & Procedures,Social Media & Misinformation,Hospitalization & Patient Outcomes
publication_year,,,,,,,,,,,,,,,
2020,55,13,22,10,5,38,56,32,36,10,17,46,53,28,29
2021,13,16,23,14,27,15,14,14,30,12,13,15,30,4,21
2022,2,2,2,1,1,3,0,0,2,1,1,2,4,0,6


topic_id,Public Health & Workforce Impact,Online Education & Training,Clinical Virology & Infection,Medical Imaging & Deep Learning,Financial Markets & Economic Impact,Mental Health & Psychological Effects,Epidemiological Modeling & Case Data,"Digital Health, Telemedicine & Blockchain",Severe Disease & Immune Response,Food Systems & Hospitality Impact,"Prevention, Masks & Environmental Effects","Policy, Governance & Public Response",Clinical Care & Procedures,Social Media & Misinformation,Hospitalization & Patient Outcomes
publication_year,,,,,,,,,,,,,,,
2020,0.122222,0.028889,0.048889,0.022222,0.011111,0.084444,0.124444,0.071111,0.080000,0.022222,0.037778,0.102222,0.117778,0.062222,0.064444
2021,0.049808,0.061303,0.088123,0.053640,0.103448,0.057471,0.053640,0.053640,0.114943,0.045977,0.049808,0.057471,0.114943,0.015326,0.080460
2022,0.074074,0.074074,0.074074,0.037037,0.037037,0.111111,0.000000,0.000000,0.074074,0.037037,0.037037,0.074074,0.148148,0.000000,0.222222


We see that epidemiological modeling and case data is very dominant in 2020 and falls off, signifying an early pandemic modeling surge. Financial markets and economic impact spikes in 2021, which aligns with delayed economic analysis after COVID impacts. Clinical Care and procsdures stay strong across all years, which is a stable backbone topic, and hospitalization and patient outcomes grows in 2022 signaling a deeper clinical focus over time. We can also see how digital health and telemedicine matches emergency adoption mechanics

In [6]:
# Load Embeddings
conn = sqlite3.connect(DB_PATH)

emb_df = pd.read_sql_query("""
    SELECT
        document_id,
        embedding
    FROM document_embeddings
    WHERE model_name = ?
""", conn, params=[EMBEDDING_MODEL_NAME])

conn.close()

# deserialize embeddings
emb_df["embedding"] = emb_df["embedding"].apply(
    lambda x: np.array(pickle.loads(x))
)

print("Loaded embeddings:", emb_df.shape)
display(emb_df.head())

Loaded embeddings: (738, 2)


,document_id,embedding
0,1,"[-0.022171408, -0.045305148, -0.035417516, 0.0..."
1,2,"[0.054407, 0.02136461, -0.021674903, 0.0654725..."
2,3,"[0.03656542, 0.027708251, 0.04394856, 0.021187..."
3,4,"[0.060955044, -0.018265247, -0.056825712, -0.0..."
4,5,"[0.097378224, -0.005816912, 0.026064621, 0.034..."


In [7]:
# Merge topics and embeddings
topic_emb_df = topic_docs_df.merge(
    emb_df,
    on="document_id",
    how="inner"
)

print("Merged topic + embeddings:", topic_emb_df.shape)
display(topic_emb_df.head())

Merged topic + embeddings: (738, 6)


,document_id,publication_year,title,topic_id,topic_label,embedding
0,1,2020,Study protocol: Comparison of different risk p...,6,Epidemiological Modeling & Case Data,"[-0.022171408, -0.045305148, -0.035417516, 0.0..."
1,2,2020,"Mental health, sleep quality and quality of li...",5,Mental Health & Psychological Effects,"[0.054407, 0.02136461, -0.021674903, 0.0654725..."
2,3,2020,Parent perspectives on food allergy management...,9,Food Systems & Hospitality Impact,"[0.03656542, 0.027708251, 0.04394856, 0.021187..."
3,4,2020,COVID-19 critical illness in Sweden: character...,14,Hospitalization & Patient Outcomes,"[0.060955044, -0.018265247, -0.056825712, -0.0..."
4,5,2020,Geolocated Twitter-based population mobility i...,13,Social Media & Misinformation,"[0.097378224, -0.005816912, 0.026064621, 0.034..."


In [8]:
# compute centroids by year
centroids = []

for (year, topic_id), group in topic_emb_df.groupby(["publication_year", "topic_id"]):
    emb_stack = np.vstack(group["embedding"].values)
    centroid = emb_stack.mean(axis=0)
    
    centroids.append({
        "publication_year": year,
        "topic_id": topic_id,
        "centroid": centroid,
        "n_docs": len(group)
    })

centroids_df = pd.DataFrame(centroids)

print("Centroids computed:", centroids_df.shape)
display(centroids_df.head())

Centroids computed: (42, 4)


,publication_year,topic_id,centroid,n_docs
0,2020,0,"[0.022808092, 0.03738438, -0.025412643, 0.0275...",55
1,2020,1,"[0.01232472, 0.029829707, 0.0024571244, 0.0058...",13
2,2020,2,"[0.010815447, 0.026126163, -0.027783157, 0.007...",22
3,2020,3,"[-0.0009460218, -0.01684355, -0.00052365696, -...",10
4,2020,4,"[-0.03659775, -0.011309656, 0.012368618, 0.014...",5


In [9]:
# add labels for readability
centroids_df["topic_label"] = centroids_df["topic_id"].map(topic_id_to_label)

display(centroids_df.head())

,publication_year,topic_id,centroid,n_docs,topic_label
0,2020,0,"[0.022808092, 0.03738438, -0.025412643, 0.0275...",55,Public Health & Workforce Impact
1,2020,1,"[0.01232472, 0.029829707, 0.0024571244, 0.0058...",13,Online Education & Training
2,2020,2,"[0.010815447, 0.026126163, -0.027783157, 0.007...",22,Clinical Virology & Infection
3,2020,3,"[-0.0009460218, -0.01684355, -0.00052365696, -...",10,Medical Imaging & Deep Learning
4,2020,4,"[-0.03659775, -0.011309656, 0.012368618, 0.014...",5,Financial Markets & Economic Impact


# Topic Drift

In [10]:
# Cosine similarity helper
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

In [11]:
# drift per topic
drift_records = []

years = sorted(centroids_df["publication_year"].unique())

for topic_id in centroids_df["topic_id"].unique():
    topic_data = centroids_df[centroids_df["topic_id"] == topic_id]
    
    for i in range(len(years) - 1):
        y1, y2 = years[i], years[i+1]
        
        c1 = topic_data[topic_data["publication_year"] == y1]
        c2 = topic_data[topic_data["publication_year"] == y2]
        
        if len(c1) == 0 or len(c2) == 0:
            continue
        
        sim = cosine_sim(
            c1["centroid"].values[0],
            c2["centroid"].values[0]
        )
        
        drift_records.append({
            "topic_id": topic_id,
            "topic_label": topic_id_to_label[topic_id],
            "year_from": y1,
            "year_to": y2,
            "cosine_similarity": sim,
            "drift": 1 - sim
        })

drift_df = pd.DataFrame(drift_records)

display(drift_df.sort_values("drift", ascending=False))

,topic_id,topic_label,year_from,year_to,cosine_similarity,drift
19,10,"Prevention, Masks & Environmental Effects",2021,2022,0.466462,0.533538
21,11,"Policy, Governance & Public Response",2021,2022,0.577730,0.422270
17,9,Food Systems & Hospitality Impact,2021,2022,0.639939,0.360061
9,4,Financial Markets & Economic Impact,2021,2022,0.652110,0.347890
16,9,Food Systems & Hospitality Impact,2020,2021,0.686920,0.313080
3,1,Online Education & Training,2021,2022,0.720548,0.279452
1,0,Public Health & Workforce Impact,2021,2022,0.756892,0.243108
15,8,Severe Disease & Immune Response,2021,2022,0.806265,0.193735
7,3,Medical Imaging & Deep Learning,2021,2022,0.812178,0.187822
8,4,Financial Markets & Economic Impact,2020,2021,0.820118,0.179882


We can see that from 2020-2021, topics appear mostly stable but then moving to 2022 there is large drift. This is consistent with what's in the news since 2020 was the initial shock, and 2021 was consolidation. Most notably:

- Prevention, Masks, and Environmental Effects had the biggest shift: the topic was reframed from mask efficacy/PPE shortages to environmental impact and ventilation. 

- In terms of policy, the drift also is quite large, as we went from lockdowns/emergency policy to government critique. 

- Food systemas and hospitality: supply chain shock to consumer response and industry adaptation.

- financial markets: COVID crash to 2021 macro recovery.

- Online education, public health, and mental health appear relatively stable over time, only changing based on context, and clinical/biomedical clusters only change in volume

In [12]:
# Making a time series per topic
topic_trajectories = {}

for topic_id in centroids_df["topic_id"].unique():
    topic_data = centroids_df[centroids_df["topic_id"] == topic_id]
    topic_data = topic_data.sort_values("publication_year")
    
    trajectory = np.vstack(topic_data["centroid"].values)
    
    topic_trajectories[topic_id] = {
        "label": topic_id_to_label[topic_id],
        "years": topic_data["publication_year"].tolist(),
        "trajectory": trajectory
    }

print("Built trajectories for", len(topic_trajectories), "topics")

Built trajectories for 15 topics


In [13]:
import pickle

output_path = PROJECT_ROOT / "data" / "topic_trajectories.pkl"

with open(output_path, "wb") as f:
    pickle.dump(topic_trajectories, f)

print("Saved trajectories to:", output_path)

Saved trajectories to: /Users/jonathanma/Desktop/DS Projects/diffusion-topic-evaluation/data/topic_trajectories.pkl
